# Images and Imagecollections

In [ ]:
import ee # Google Earth Engine Python Client
import geemap # frontend του ee Google Earth Engine
import pprint # pretty print')
from pathlib import Path # διαχείριση αρχείων
from osgeo import gdal  # για την ανάγνωση και εγγραφή γεωχωρικών δεδομένων
import glob # για την εύρεση αρχείων με συγκεκριμένη επέκταση
import os # για την διαχείριση περιβάλλοντος και αρχείων
from dotenv import load_dotenv # για την φόρτωση μεταβλητών περιβάλλοντος από .env αρχεία

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Colab\ Notebooks/gee
except:
  print("Παρακαλώ εκτελέστε το notebook στο Google Colab")

In [ ]:

# Load environment variables from .env file
load_dotenv()
project_id = os.getenv('project_id')


# gee authentication
ee.Authenticate()
ee.Initialize(project=project_id)



OUTPUT_DIR = Path("output")


In [ ]:
#  Φόρτωση εικόνας SRTM
image = ee.Image("CGIAR/SRTM90_V4")
display(image)

In [ ]:
#  Λήψη του προβολικού συστήματος της εικόνας
# Σημείωση: Επιλέγουμε ένα κανάλι (band), καθώς διαφορετικά κανάλια 
# μπορεί θεωρητικά να έχουν διαφορετικές αναλύσεις
projection = image.select(0).projection()

# Λήψη της ονομαστικής (nominal) ανάλυσης (σε μέτρα)
# Αυτό επιστρέφει το μέγεθος του pixel (συνήθως μέτρα)
resolution = projection.nominalScale().getInfo()

# Λήψη του γεωγραφικού σύστηματος αναφοράς (CRS) και τις παραμέτρους μετασχηματισμού
crs = projection.crs().getInfo()
transform = projection.transform().getInfo()

print(f"Σύστημα Προβολής (CRS): {crs}")
print(f"Ονομαστική Ανάλυση: {resolution} μέτρα")
print(f"Πίνακας Μετασχηματισμού (Transform): {transform}")

In [ ]:
# Λήψη και εκτύπωση πληροφοριών για την εικόνα
image_info = image.getInfo()

print("Τύπος εικόνας:", image_info['type'])
print("ID εικόνας:", image_info['id'])
print("Πλήθος bands:", len(image_info['bands']))

image_info

In [ ]:
# Μεταδεδομένα (properties) της εικόνας
property_names = list(image_info.get('properties', {}).keys())
pprint.pprint(property_names)

In [ ]:
m = geemap.Map()

terrain_palette = [
    '#006633', 
    '#229954', 
    '#7DCEA0', 
    '#F7DC6F',
    '#B8860B', 
    '#7E5109',
    '#5D4037', 
    '#FFFFFF' 
]

dem_vis = {
    'bands': 'elevation',
    'min': 0,
    'max': 4500,
    'palette': terrain_palette
}

m.addLayer(image, dem_vis, 'SRTM DEM')
m.centerObject(image, 5)
m

# Masking εικόνα

In [ ]:
# Δημιουργία μιας δυαδικής (binary) μάσκας όπου elevation > 3000 μετατρέπεται σε 1 (True) και τα υπόλοιπα σε 0 (False)
mask = image.select('elevation').gt(3000).rename('mask')
# mask.bandNames().getInfo()

# Ενημέρωση της εικόνας προελεύσεως με τη μάσκα
# μόνο τα pixels όπου mask=1 (True) θα είναι ορατά στην τελική εικόνα, ενώ τα υπόλοιπα θα γίνουν διαφανή
filtered_dem = image.updateMask(mask)

m.addLayer(filtered_dem, {'min': 3000, 'max': 5000, 'palette': ['blue', 'white']}, 'Elevation > 3000m')
m.addLayer(mask, {'min': 0, 'max': 1,'palette': ['black', 'red']}, 'Binary Mask (Red=Keep)', False, 0.5)

# Hillshade, slope, aspect

Για τον υπολογισμό της κλίσης (Slope), την έκθεση (Aspect) και την φωτοσκίαση (Hillshade), χρησιμοποιούμε την  συνάρτηση `ee.Terrain.products()`.

In [ ]:

# Υπολογισμός Παραγώγων Εδάφους
# Η συνάρτηση products() υπολογίζει ταυτόχρονα κλίση (slope), προσανατολισμό (aspect) και ορθοφωτοσκίαση (hillshade)
terrain = ee.Terrain.products(image)


# Δημιουργία Διαδραστικού Χάρτη
Map = geemap.Map()

# Ορθοφωτοσκίαση (Hillshade): Συνήθως σε κλίμακα του γκρι (0-255)
Map.addLayer(terrain.select('hillshade'), {'min': 0, 'max': 255}, 'Ορθοφωτοσκίαση')

# Κλίση (Slope): Σε μοίρες (0-90). Πράσινο για επίπεδα, κόκκινο για απότομα.
Map.addLayer(terrain, {'bands': ['slope'],'min': 0, 'max': 45, 'palette': ['green', 'yellow', 'red']}, 'Κλίση (Μοίρες)')

# Έκθεση (Aspect): Σε μοίρες (0-360). Δείχνει τον προσανατολισμό του εδάφους.
Map.addLayer(terrain.select('aspect'), {'min': 0, 'max': 360, 'palette': ['#0000ff', '#00ff00', '#ffff00', '#ff0000']}, 'Έκθεση (Προσανατολισμός)')

Map.centerObject(image, 2)

aspect_legend = {
    'Βορράς (N)': '#0000ff',
    'Ανατολή (E)': '#00ff00',
    'Νότος (S)': '#ffff00',
    'Δύση (W)': '#ff0000'
}

Map.add_legend(title="Προσανατολισμός Πλαγιάς", legend_dict=aspect_legend, position='bottomright')

Map

In [ ]:
nuts_level2 = ee.FeatureCollection('projects/civic-meridian-417810/assets/NUTS_RG_01M_2021_4326_LEVL_2')

# Αποκοπή της εικόνας με βάση τα όρια των πολυγώνων NUTS2
terrain_nuts2 = terrain.clip(nuts_level2)

Map.addLayer(terrain_nuts2, dem_vis, 'Terrain NUTS2')
Map.addLayer(nuts_level2, {'color': 'green'}, 'NUTS2')


# Φιλτράρισμα για μια συγκεκριμένη περιοχή (π.χ. Αττική - EL30)
attiki_region = nuts_level2.filter(ee.Filter.eq('NUTS_ID', 'EL30'))

# Αποκοπή της εικόνας
terrain_attiki = image.clip(attiki_region)

# Προσθήκη στον χάρτη
Map.addLayer(terrain_attiki, {'min': 0, 'max': 3000}, 'Terrain Attiki')
Map.centerObject(attiki_region, 8)

In [ ]:
# Εξαγωγή της εικόνας σε GeoTIFF
export_params = {
    'image': image,
    'description': 'terrain_attiki',
    'scale': 90,
    'region': attiki_region.geometry(), # Πολύ σημαντικό!
    'fileFormat': 'GeoTIFF',
    'folder' :'GEE_Exports',          # Το όνομα του φακέλου στο Google Drive σας
    'crs': 'EPSG:2100',   # Ελληνικό Γεωδετικό Σύστημα Αναφοράς 1987
    'maxPixels': 1e13,    # Μέγιστος αριθμός pixels για αποφυγή σφαλμάτων
}
ee.batch.Export.image.toDrive(**export_params).start()

Η παράμετρος maxPixels στο Google Earth Engine είναι μια "δικλείδα ασφαλείας". Ο σκοπός της είναι να προστατεύσει τους διακομιστή της Google από υπερβολικά μεγάλα αιτήματα επεξεργασίας που θα μπορούσαν να εξαντλήσουν τους πόρους του συστήματος.

Όταν ζητάς μια εξαγωγή (Export) ή έναν υπολογισμό, το GEE πολλαπλασιάζει την έκταση της περιοχής σου (Region) με την ανάλυση (Scale) που όρισες. Αν ο συνολικός αριθμός των pixels υπερβαίνει το όριο του maxPixels, η εργασία θα σταματήσει με σφάλμα.
1. Γιατί χρειαζόμαστε το maxPixels;

Από προεπιλογή, το όριο είναι συνήθως 100.000.000 (10^8) pixels. Αν και ακούγεται μεγάλο νούμερο, για δορυφορικά δεδομένα υψηλής ανάλυσης είναι πολύ εύκολο να ξεπεραστεί:

- Ένα τετράγωνο 10 km×10 km με ανάλυση Sentinel-2 (10 m) έχει 1.000.000 pixels.

- Αν προσπαθήσεις να εξάγεις ολόκληρη την Ελλάδα σε ανάλυση 30 m, θα ξεπεράσεις κατά πολύ το προεπιλεγμένο όριο.

2. Πώς να το ρυθμίσεις σωστά

Αν λάβεις το σφάλμα "Export too large. (Max pixels exceeded)", πρέπει να αυξήσεις την τιμή. Συνήθως χρησιμοποιούμε επιστημονική σημείωση για ευκολία:

- 1e8 = 100 εκατομμύρια (Προεπιλογή)

- 1e9 = 1 δισεκατομμύριο

- 1e12 ή 1e13 = Πολύ μεγάλες εξαγωγές (π.χ. ολόκληρες χώρες σε υψηλή ανάλυση)

Η αύξηση του maxPixels δεν κάνει την επεξεργασία πιο γρήγορη. Απλώς επιτρέπει στο GEE να ξεκινήσει μια εργασία που απαιτεί πολύ χρόνο. Αν το παρακάνεις:

- Η εξαγωγή μπορεί να διαρκέσει πολλές ώρες ή και μέρες.

- Το αρχείο GeoTIFF στο Google Drive μπορεί να είναι τεράστιο (πολλά GB), καθιστώντας δύσκολο το άνοιγμά του στο QGIS ή το ArcGIS του υπολογιστή σου.

Αν δεις ότι χρειάζεσαι maxPixels πάνω από 1e12, ίσως είναι καλύτερο να αυξήσεις το scale (π.χ. από 30 σε 100 μέτρα) αν η ακρίβεια δεν είναι το απόλυτο ζητούμενο, ή να χωρίσεις την περιοχή σου σε μικρότερα κομμάτια.

In [ ]:
# Εναλλακτικά, για απευθείας λήψη (download) της εικόνας ως GeoTIFF, συνήθως για μικρότερες περιοχές λόγω περιορισμών μεγέθους

geemap.download_ee_image(terrain_attiki,
                          region=attiki_region.geometry().bounds(), 
                          filename= OUTPUT_DIR / "attiki.tif", 
                          scale=500)



In [ ]:
# bounding box της Αττικής
bbox = attiki_region.bounds()
Map.addLayer(bbox, {}, "bbox")

# Δημιουργία Fishnet (Grid) πάνω από την Αττική
# h_interval και v_interval καθορίζουν το μέγεθος των κελιών του grid σε μοίρες (π.χ. 0.3° x 0.3°), 
# delta είναι ένα μικρό buffer για να αποφευχθούν προβλήματα με τα όρια
fishnet = geemap.fishnet(bbox, h_interval=0.3, 
    v_interval=0.3, 
    delta=1)


Map.addLayer(fishnet, {'color': "blue"}, "fishnet")
Map

In [ ]:
# ληψη των tiles της εικόνας με βάση το fishnet
geemap.download_ee_image_tiles_parallel(
    image, fishnet, out_dir=OUTPUT_DIR / "tiles", scale=500, crs="EPSG:2100"
)


Για να συλλέγουμε τα επιμέρους tiles σε ένα ακέραιο αρχείο raster μπορούμε να χρησιμοποιήσουμε την command line εντολή από το directory των tiffs:

```
gdalbuildvrt attiki.vrt *.tif
```
ή με python όπως παρακάτω. Απαιτείται η εγκατάσταση του πακέτου gdal στο virtual enviroment:

`mamba install gdal` ή

`conda install conda-forge::gdal`


In [ ]:

# ανάκτηση αρχείων .tif που δημιουργήθηκαν από την προηγούμενη διαδικασία
input_files = glob.glob(str(OUTPUT_DIR / 'tiles' / "*.tif"))

# ορισμός του ονόματος του αρχείου VRT που θα δημιουργηθεί
output_vrt = OUTPUT_DIR /  'tiles'/ "attiki.vrt"

# δημιουργία του VRT αρχείου από τα tiles
# Η 'options' παράμετρος μπορεί να πάρει οποιοδήποτε όρισμα βρίσκεται στην CLI έκδοση του gdalbuildvrt
vrt_options = gdal.BuildVRTOptions(resampleAlg='near', addAlpha=False)
gdal.BuildVRT(output_vrt, input_files, options=vrt_options)

print(f"Επιτυχής δημιουργία {output_vrt} από {len(input_files)} αρχεία.")


Ανάλογα είναι εφικτό να εξαχθεί ανά tile το geotiff στο Google Drive

In [ ]:
# Μετατροπή του FeatureCollection σε λίστα για να τρέξουμε το loop
features = fishnet.getInfo()['features']

for i, feature in enumerate(features):
    geom = ee.Feature(feature).geometry()
    
    task = ee.batch.Export.image.toDrive(
        image=image.clip(geom),
        description=f'Attica_Part_{i}',
        folder='GEE_Exports',
        region=geom,
        scale=500,
        crs='EPSG:2100',
        maxPixels=1e8
    )
    task.start()
    print(f'Ξεκίνησε η εξαγωγή για το κελί {i}')

In [ ]:

# Ορισμός της Συλλογής και Φιλτράρισμα
L8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
              .filterBounds(bbox) # Φιλτράρισμα με βάση τα όρια της Αττικής
              .filterDate('2024-03-01', '2025-03-01') # Φιλτράρισμα με βάση το εύρος ημερομηνιών
              .filter(ee.Filter.lt('CLOUD_COVER', 10)) # Φιλτράρισμα με βάση το ποσοστό νεφοκάλυψης (Cloud Cover < 10%
           ) 

# Έλεγχος αν η συλλογή είναι κενή πριν προχωρήσουμε

# Εκτύπωση του αριθμού των εικόνων (Size)
count = L8.size().getInfo()
print(f"Συνολικός αριθμός εικόνων στη συλλογή: {count}")



In [ ]:


# πρώτη εικόνα για προβολή όλων των διαθέσιμων ιδιότητων (Metadata keys)
first_image = L8.first()
props = first_image.propertyNames().getInfo()
print("\nΔιαθέσιμες Ιδιότητες (Metadata Properties):")
pprint.pprint(props)

# Εκτύπωση συγκεκριμένων ιδιοτήτων για ΟΛΕΣ τις εικόνες της συλλογής
# Παίρνουμε τα ID και το ποσοστό νεφοκάλυψης σε λίστες
ids = L8.aggregate_array('LANDSAT_PRODUCT_ID').getInfo() 
clouds = L8.aggregate_array('CLOUD_COVER').getInfo()
dates = L8.aggregate_array('DATE_ACQUIRED').getInfo()

print("\nΛίστα Εικόνων στη Συλλογή:")
for i in range(count):
    print(f"[{i+1}] ID: {ids[i]} | Ημερομηνία: {dates[i]} | Σύννεφα: {clouds[i]}%")


In [ ]:
# Φιλτράρισμα για εικόνες με νεφοκάλυψη < 1%
clear_L8 = L8.filter(ee.Filter.lt('CLOUD_COVER', 1))

print(f"Συνολικός αριθμός εικόνων στη συλλογή: {L8.size().getInfo()}")
print(f"Συνολικός αριθμός εικόνων στη συλλογή (χαμηλότερη νεφοκάλυψη): {clear_L8.size().getInfo()}")

# Scaling τιμών και δημιουργία εικόνας NDVI

In [ ]:
# Υπολογισμός του διάμεσου (median) για κάθε pixel σε όλες τις εικόνες της συλλογής
median_L8 = clear_L8.median()

# συνάρτηση για εφαρμογή των συντελεστών (scale factors) για τα οπτικά και θερμικά κανάλια
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )

# Εφαρμογή των συντελεστών κλίμακας (scale factors) στην εικόνα του διάμεσου
L8_scaled = apply_scale_factors(median_L8)

# 4. Υπολογισμός NDVI
# Band 5 = NIR, Band 4 = Red
ndvi = L8_scaled.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
ndvi = ndvi.clip(attiki_region) # Περιορισμός στην περιοχή ενδιαφέροντος

# 5. Εμφάνιση αποτελεσμάτων
m = geemap.Map()

# Παλέτα για NDVI (από καφέ/κίτρινο σε βαθύ πράσινο)
ndvi_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#FFFFFF', '#CE7E45', '#DF923D', '#F1B555', '#FCD163', 
                '#99B718', '#74A401', '#3E8601', '#207401', '#056201']
}

# True Color Image (Bands 4, 3, 2) με κλίμακα 0-0.3 για καλύτερη εμφάνιση
m.addLayer(L8_scaled, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.2}, 'True Color Image L8')
m.addLayer(ndvi, ndvi_vis, 'NDVI Single Image')

m.centerObject(attiki_region, 8)

m